# Create and run a local RAG pipeline from scratch

## Importing PDF from scratch

In [2]:
import os
import requests

# Get pdf path
pdf_path = "human_nutrition_text"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesnt exist, downloading...")

    # Enter the URL of the PDF
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
    
    # The local filename to save the downloaded file
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")
else:
    print(f"File {pdf_path} exists")



File human_nutrition_text exists


In [3]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text:str) -> str:
    """ Performs minor formatting on text. """
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path : str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text)
        pages_and_texts.append({"page_number":page_number - 41, 
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4, 
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path = pdf_path)
pages_and_texts[:2]

                            
        

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [4]:
import random

random.sample(pages_and_texts, k=5)

[{'page_number': 422,
  'page_char_count': 1710,
  'page_word_count': 282,
  'page_sentence_count_raw': 16,
  'page_token_count': 427.5,
  'text': 'turnover rate. During exercise, especially when it is performed for  longer than two to three hours, muscle tissue is broken down and  some of the amino acids are catabolized to fuel muscle contraction.  To avert excessive borrowing of amino acids from muscle tissue to  synthesize energy during prolonged exercise, protein needs to be  obtained from the diet. Intense exercise, such as strength training,  stresses muscle tissue so that afterward, the body adapts by  building bigger, stronger, and healthier muscle tissue. The body  requires protein post-exercise to accomplish this. The IOM does  not set different RDAs for protein intakes for athletes, but the AND,  the American College of Sports Medicine, and Dietitians of Canada  have the following position statements4:  Nitrogen balance studies suggest that dietary protein intake  necessary 

In [6]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [7]:
df.describe()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00000,1208.000000,1208.000000,1208.000000,1208.000000
mean,562.50000,1148.004139,199.499172,10.519868,287.001035
std,348.86387,560.382275,95.830681,6.548495,140.095569
min,-41.00000,0.000000,1.000000,1.000000,0.000000
25%,260.75000,762.000000,134.000000,5.000000,190.500000
50%,562.50000,1231.500000,216.000000,10.000000,307.875000
75%,864.25000,1603.500000,272.000000,15.000000,400.875000
max,1166.00000,2308.000000,430.000000,39.000000,577.000000


### Further text processing (splitting pages into sentences)

Two ways to do this : 
1. Do this by splitting on `". "`.
2. We can do this with a NLP library like Spacy or NLTK

In [23]:
from spacy.lang.en import English

nlp = English()

# Add a sentencizer pipeline
nlp.add_pipe("sentencizer")

# Create document instance as an example
doc = nlp("This is a sentence. This is another sentence. I like potatoes.")
assert len(list(doc.sents)) == 3

# Print out sentences split
list(doc.sents)


[This is a sentence., This is another sentence., I like potatoes.]

In [26]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # make sure all sentences are strings. The default type is spacy datatype
    item["sentence"] = [str(sentence) for sentence in item["sentences"]]

    # count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [27]:
random.sample(pages_and_texts, k=1)

[{'page_number': 1045,
  'page_char_count': 308,
  'page_word_count': 53,
  'page_sentence_count_raw': 3,
  'page_token_count': 77.0,
  'text': 'effectively. By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient deficiencies,  ward off chronic disease, and  bolstering a sense of overall health  and well-being.  Introduction  |  1045',
  'sentences': [effectively.,
   By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient deficiencies,  ward off chronic disease, and  bolstering a sense of overall health  and well-being.,
    Introduction  |  1045],
  'sentence': ['effectively.',
   'By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient d

In [28]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00


### Chunking our sentences together 

The concept of splitting larger pieces of text into smaller ones is often referred to as text splitting or chunking.
    We'll split into groups of 10 sentences.